# import libarires

In [51]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,confusion_matrix,ConfusionMatrixDisplay ,classification_report , roc_auc_score
from feature_engine.imputation import AddMissingIndicator, MeanMedianImputer
from feature_engine.discretisation import EqualFrequencyDiscretiser
from feature_engine.encoding import WoEEncoder

# import dataset

In [52]:
df = pd.read_csv("../01_data/credit_risk_clean.csv")
df.head()

,age,annual_income,home_ownership,employment_years,loan_purpose,credit_grade,loan_amount,interest_rate,default_flag,loan_income_ratio,prior_default_flag,credit_history_years
0,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
1,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
2,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
3,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4
4,21,9900,OWN,2.0,VENTURE,A,2500,7.14,1,0.25,N,2


In [53]:
numeric_columns = [
    "age", "annual_income", "employment_years", "loan_amount",
    "interest_rate", "loan_income_ratio", "credit_history_years",
]

categorical_columns = [
    "home_ownership", "loan_purpose", "credit_grade", "prior_default_flag",
]

all_features = numeric_columns + categorical_columns

In [54]:
X = df[all_features]
y = df['default_flag']

# train test split

In [55]:
X_train, X_test , y_train ,y_test = train_test_split(
    X,y,         # feature and target
    test_size = 0.2,
    stratify= y,
    random_state = 42
)

In [56]:
print("Shape of Training data is :", X_train.shape,y_train.shape)
print("Shape of Testing data  is :", X_test.shape,y_test.shape)

Shape of Training data is : (25927, 11) (25927,)
Shape of Testing data  is : (6482, 11) (6482,)


In [57]:
# Q: How many defautls and non defaults in training and testing data 
print("Training data \n", y_train.value_counts(normalize=True).to_dict())
print("Testing data \n" ,y_test.value_counts(normalize=True).to_dict())

Training data 
 {0: 0.7813090600532263, 1: 0.21869093994677363}
Testing data 
 {0: 0.781240357914224, 1: 0.218759642085776}


# misssing values

In [58]:
X_train.isna().sum()

age                        0
annual_income              0
employment_years         710
loan_amount                0
interest_rate           2463
loan_income_ratio          0
credit_history_years       0
home_ownership             0
loan_purpose               0
credit_grade               0
prior_default_flag         0
dtype: int64

In [59]:
X_test.isna().sum()

age                       0
annual_income             0
employment_years        177
loan_amount               0
interest_rate           631
loan_income_ratio         0
credit_history_years      0
home_ownership            0
loan_purpose              0
credit_grade              0
prior_default_flag        0
dtype: int64

In [60]:
missing_cols = X_train[numeric_columns].columns[X_train[numeric_columns].isna().any()].tolist()
missing_cols

['employment_years', 'interest_rate']

In [61]:
missing_indicator = AddMissingIndicator(variables=missing_cols)
median_imputer = MeanMedianImputer(imputation_method='median',variables=missing_cols)


In [62]:
# Step 1: Add missing indicator columns
# - fit_transform on Train to learn and transform
# - transform on Test using the parameters learned from Train
X_train = missing_indicator.fit_transform(X_train)
X_test = missing_indicator.transform(X_test)

# Step 2: Impute missing values with medians
# - fit_transform on Train to calculate medians and fill missing values
# - transform on Test to fill missing values using X_train medians
X_train = median_imputer.fit_transform(X_train)
X_test = median_imputer.transform(X_test)

In [63]:
X_train.isna().sum()

age                     0
annual_income           0
employment_years        0
loan_amount             0
interest_rate           0
loan_income_ratio       0
credit_history_years    0
home_ownership          0
loan_purpose            0
credit_grade            0
prior_default_flag      0
employment_years_na     0
interest_rate_na        0
dtype: int64

In [64]:
X_test.isna().sum()

age                     0
annual_income           0
employment_years        0
loan_amount             0
interest_rate           0
loan_income_ratio       0
credit_history_years    0
home_ownership          0
loan_purpose            0
credit_grade            0
prior_default_flag      0
employment_years_na     0
interest_rate_na        0
dtype: int64

In [65]:
indicator_cols = [c for c in X_train.columns if c.endswith('_na')]
indicator_cols

['employment_years_na', 'interest_rate_na']

In [66]:
X_train

,age,annual_income,employment_years,loan_amount,interest_rate,loan_income_ratio,credit_history_years,home_ownership,loan_purpose,credit_grade,prior_default_flag,employment_years_na,interest_rate_na
31112,37,85000,19.0,10000,9.99,0.12,16,MORTGAGE,HOMEIMPROVEMENT,B,N,0,0
5364,25,50000,7.0,8500,15.70,0.17,4,RENT,EDUCATION,E,N,0,0
13349,26,90000,1.0,20000,10.99,0.22,3,MORTGAGE,PERSONAL,B,N,0,1
31695,41,88712,2.0,16000,10.95,0.18,15,RENT,VENTURE,B,N,0,0
24075,35,39000,2.0,10000,13.16,0.26,6,RENT,MEDICAL,C,N,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
32122,38,110000,5.0,16000,11.99,0.15,13,MORTGAGE,MEDICAL,B,N,0,0
18667,30,14800,5.0,1200,8.00,0.08,9,RENT,DEBTCONSOLIDATION,A,N,0,0
30477,39,60000,3.0,8000,9.88,0.13,14,RENT,PERSONAL,B,N,0,0
17957,30,180000,2.0,12500,6.03,0.07,6,MORTGAGE,DEBTCONSOLIDATION,A,N,0,0


# quantile bining
Convert Numerical Variable ---> Categorical         
splitting data into equal-sized groups(categories)               
$I_k = \left[ Q\left(\frac{k-1}{q}\right), Q\left(\frac{k}{q}\right) \right)$


In [69]:
quantile_binner = EqualFrequencyDiscretiser(
    q=10, 
    variables=numeric_columns, 
    return_object=True
)
X_train = quantile_binner.fit_transform(X_train)
X_test = quantile_binner.transform(X_test)

In [78]:
X_train

,age,annual_income,employment_years,loan_amount,interest_rate,loan_income_ratio,credit_history_years,home_ownership,loan_purpose,credit_grade,prior_default_flag,employment_years_na,interest_rate_na
31112,9,7,8,5,3,3,6,MORTGAGE,HOMEIMPROVEMENT,B,N,0,0
5364,3,4,6,5,9,5,1,RENT,EDUCATION,E,N,0,0
13349,4,8,0,9,4,7,0,MORTGAGE,PERSONAL,B,N,0,1
31695,9,8,1,8,4,5,6,RENT,VENTURE,B,N,0,0
24075,8,2,1,5,7,8,3,RENT,MEDICAL,C,N,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
32122,9,8,4,8,6,4,6,MORTGAGE,MEDICAL,B,N,0,0
18667,7,0,4,0,2,1,4,RENT,DEBTCONSOLIDATION,A,N,0,0
30477,9,5,2,4,3,4,6,RENT,PERSONAL,B,N,0,0
17957,7,9,1,7,0,1,3,MORTGAGE,DEBTCONSOLIDATION,A,N,0,0


In [71]:
# Define full encoding column list including categoricals and missing indicators
all_encode_columns = numeric_columns + categorical_columns
all_encode_columns

['age',
 'annual_income',
 'employment_years',
 'loan_amount',
 'interest_rate',
 'loan_income_ratio',
 'credit_history_years',
 'home_ownership',
 'loan_purpose',
 'credit_grade',
 'prior_default_flag']

In [72]:
all_encode_columns

['age',
 'annual_income',
 'employment_years',
 'loan_amount',
 'interest_rate',
 'loan_income_ratio',
 'credit_history_years',
 'home_ownership',
 'loan_purpose',
 'credit_grade',
 'prior_default_flag']

In [73]:
X_train

,age,annual_income,employment_years,loan_amount,interest_rate,loan_income_ratio,credit_history_years,home_ownership,loan_purpose,credit_grade,prior_default_flag,employment_years_na,interest_rate_na
31112,9,7,8,5,3,3,6,MORTGAGE,HOMEIMPROVEMENT,B,N,0,0
5364,3,4,6,5,9,5,1,RENT,EDUCATION,E,N,0,0
13349,4,8,0,9,4,7,0,MORTGAGE,PERSONAL,B,N,0,1
31695,9,8,1,8,4,5,6,RENT,VENTURE,B,N,0,0
24075,8,2,1,5,7,8,3,RENT,MEDICAL,C,N,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
32122,9,8,4,8,6,4,6,MORTGAGE,MEDICAL,B,N,0,0
18667,7,0,4,0,2,1,4,RENT,DEBTCONSOLIDATION,A,N,0,0
30477,9,5,2,4,3,4,6,RENT,PERSONAL,B,N,0,0
17957,7,9,1,7,0,1,3,MORTGAGE,DEBTCONSOLIDATION,A,N,0,0


In [74]:
# Cast all features to category dtype for feature-engine WoE compliance
for col in all_encode_columns:
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')

In [80]:
X_train

,age,annual_income,employment_years,loan_amount,interest_rate,loan_income_ratio,credit_history_years,home_ownership,loan_purpose,credit_grade,prior_default_flag,employment_years_na,interest_rate_na
31112,9,7,8,5,3,3,6,MORTGAGE,HOMEIMPROVEMENT,B,N,0,0
5364,3,4,6,5,9,5,1,RENT,EDUCATION,E,N,0,0
13349,4,8,0,9,4,7,0,MORTGAGE,PERSONAL,B,N,0,1
31695,9,8,1,8,4,5,6,RENT,VENTURE,B,N,0,0
24075,8,2,1,5,7,8,3,RENT,MEDICAL,C,N,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
32122,9,8,4,8,6,4,6,MORTGAGE,MEDICAL,B,N,0,0
18667,7,0,4,0,2,1,4,RENT,DEBTCONSOLIDATION,A,N,0,0
30477,9,5,2,4,3,4,6,RENT,PERSONAL,B,N,0,0
17957,7,9,1,7,0,1,3,MORTGAGE,DEBTCONSOLIDATION,A,N,0,0


In [82]:
X_train.drop(columns = indicator_cols,inplace = True)
X_test.drop(columns = indicator_cols,inplace = True)

# Weight of Evidence (WoE)
Weight of Evidence (WoE) transforms categorical values into continuous numeric values representing the natural log-odds of the target outcome.          
$\text{WoE}_i = \ln \left( \frac{\% \text{ Good}_i}{\% \text{ Bad}_i} \right) = \ln \left( \frac{\text{Good}_i / \text{Good}_{\text{total}}}{\text{Bad}_i / \text{Bad}_{\text{total}}} \right)$             
Log of a number > 1 means positive value. If less than 1, it means negative value.


In [83]:
woe_encoder = WoEEncoder(variables=all_encode_columns)
X_train_woe = woe_encoder.fit_transform(X_train, y_train)
X_test_woe = woe_encoder.transform(X_test)

In [84]:
X_train_woe

,age,annual_income,employment_years,loan_amount,interest_rate,loan_income_ratio,credit_history_years,home_ownership,loan_purpose,credit_grade,prior_default_flag
31112,-0.050646,-0.612238,-0.380067,-0.132917,-0.401392,-0.698532,-0.027408,-0.668658,0.254669,-0.352006,-0.203381
5364,0.021899,-0.121454,-0.243211,-0.132917,1.668766,-0.232852,0.038704,0.504774,-0.297176,1.911326,-0.203381
13349,-0.086282,-1.002990,0.303641,0.604121,-0.152715,-0.144767,0.064631,-0.668658,-0.133226,-0.352006,-0.203381
31695,-0.050646,-1.002990,0.217038,0.356750,-0.152715,-0.232852,-0.027408,0.504774,-0.436693,-0.352006,-0.203381
24075,-0.048297,0.090155,0.217038,-0.132917,-0.054267,0.665435,-0.109207,0.504774,0.251817,-0.092992,-0.203381
...,...,...,...,...,...,...,...,...,...,...,...
32122,-0.050646,-1.002990,-0.187253,0.356750,-0.172345,-0.618892,-0.027408,-0.668658,0.251817,-0.352006,-0.203381
18667,-0.127555,1.124388,-0.187253,-0.039106,-0.614725,-0.690819,-0.073924,0.504774,0.354674,-0.921311,-0.203381
30477,-0.050646,-0.236043,-0.084812,-0.300796,-0.401392,-0.618892,-0.027408,0.504774,-0.133226,-0.352006,-0.203381
17957,-0.127555,-1.014332,0.217038,0.203118,-1.171279,-0.690819,-0.109207,-0.668658,0.354674,-0.921311,-0.203381


In [85]:
X_test_woe

,age,annual_income,employment_years,loan_amount,interest_rate,loan_income_ratio,credit_history_years,home_ownership,loan_purpose,credit_grade,prior_default_flag
16587,0.238978,0.055234,-0.084812,-0.156212,-0.355929,-0.698532,0.038704,0.504774,-0.133226,-0.092992,0.739693
26852,-0.127555,-1.002990,-0.380067,0.604121,-0.054267,0.665435,-0.043488,-0.668658,0.354674,-0.092992,-0.203381
19721,-0.127555,0.090155,-0.187253,-0.102195,-0.401392,0.665435,-0.043488,0.504774,0.251817,-0.352006,-0.203381
24055,-0.127555,0.055234,0.112449,-0.132917,-0.152715,-0.144767,-0.090069,0.504774,-0.297176,-0.352006,-0.203381
24943,-0.048297,0.893621,0.112449,-0.102195,-0.401392,2.174970,-0.090069,0.504774,0.251817,-0.352006,-0.203381
...,...,...,...,...,...,...,...,...,...,...,...
7637,-0.007983,1.124388,0.303641,-0.300796,-1.171279,0.665435,0.038704,0.504774,0.251817,-0.921311,-0.203381
10310,-0.007983,-0.236043,-0.243211,0.356750,-0.355929,0.665435,0.038704,0.504774,0.354674,-0.352006,-0.203381
2328,-0.016770,0.893621,-0.178413,-0.039106,-1.171279,-0.690819,0.038704,-1.255080,-0.297176,-0.921311,-0.203381
28150,-0.048331,-0.121454,0.112449,-0.300796,-0.152715,-0.618892,-0.073924,0.504774,-0.133226,-0.352006,-0.203381


# information value(IV)
Information Value (IV) quantifies the overall predictive power of a categorical feature to separate binary outcomes (such as "Good" vs. "Bad" or "Default" vs. "Non-default"). While Weight of Evidence (WoE) measures predictive power at the individual bin level, IV aggregates WoE across all bins into a single metric for the entire feature.
$$IV = \sum_{i=1}^{k} \left( \% \text{Goods}_i - \% \text{Bads}_i \right) \times \text{WoE}_i$$

Where:

* $\% \text{Goods}_i = \frac{\text{Goods in bin } i}{\text{Total Goods}}$
* $\% \text{Bads}_i = \frac{\text{Bads in bin } i}{\text{Total Bads}}$
* $\text{WoE}_i = \ln\left(\frac{\% \text{Goods}_i}{\% \text{Bads}_i}\right)$

In [ ]:
def calculate_iv(binned_col, target):
    df_iv = pd.DataFrame({'bin': binned_col, 'target': target})
    grouped = df_iv.groupby('bin', observed=True)['target'].agg(['count', 'sum'])
    grouped.columns = ['total', 'bad']
    grouped['good'] = grouped['total'] - grouped['bad']
    grouped['pct_good'] = (grouped['good'] / grouped['good'].sum()).replace(0, 0.0001)
    grouped['pct_bad'] = (grouped['bad'] / grouped['bad'].sum()).replace(0, 0.0001)
    grouped['woe'] = np.log(grouped['pct_good'] / grouped['pct_bad'])
    grouped['iv'] = (grouped['pct_good'] - grouped['pct_bad']) * grouped['woe']
    return grouped['iv'].sum()

iv_results = {col: calculate_iv(X_train[col], y_train) for col in all_encode_columns}
iv_table = pd.DataFrame(
    sorted(iv_results.items(), key=lambda x: x[1], reverse=True),
    columns=['Variable', 'IV']
)
print("\n=== Full IV Ranking ===")
print(iv_table)


=== Full IV Ranking ===
                Variable        IV
0           credit_grade  0.882084
1      loan_income_ratio  0.877676
2          interest_rate  0.652975
3          annual_income  0.467656
4         home_ownership  0.381460
5     prior_default_flag  0.148590
6           loan_purpose  0.090643
7            loan_amount  0.087451
8       employment_years  0.057001
9                    age  0.012516
10  credit_history_years  0.004444
